# Interactive masking and processing of EELS Spectrum Images

Interactive notebook to process masking, analysis and plotting of EELS data results

Author: Ella Kitching  
Dependencies: See `requirements.txt`  
Please run cells sequentially!

Outputs:
* Edge/bulk spectra comparison
* Ce$^{3+}$ to Ce$^{4+}$ ratio maps
* Fluence-dependent oxidation state tracking

### imports

In [ ]:
import sys, os, re, glob
print('Python version:', sys.version.split('\n')[0])
print('Notebook path:', os.path.abspath('.'))

import numpy as np

import pandas as pd

import hyperspy.api as hs

from scipy.ndimage import gaussian_filter, uniform_filter, uniform_filter1d, gaussian_filter1d, median_filter, distance_transform_edt
import scipy.ndimage as ndi 
from scipy.signal import find_peaks
from scipy.interpolate import interp1d

from skimage.morphology import remove_small_objects
from skimage.restoration import denoise_nl_means, estimate_sigma
from skimage.transform import resize
from skimage.filters import threshold_li

import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.patheffects as pe
import matplotlib.gridspec as gridspec

%matplotlib qt5

In [ ]:
# make sure youve downloaded the functions file in the repo and append the path to there!
sys.path.append(r'path/to/functions/file')

from eels_processing_functions import (
    create_distance_masks, save_figure, add_scalebar,
    extract_edge_spectrum, mlls_fit, compute_m5_shift,
    cumulative_eels, normalise_spectra, remove_spectral_spikes,
    smooth_map, calculate_dose
)

### file loading and parameter set up

Load the EELS files you want to process and set up any metadata needed here.

This section also sets up file save names and save paths.

In [ ]:
# make this True if you want to automatically save! best to enable after running through first - saving can stop pop ups of the graphs.
SAVE_FIGURES = False

In [ ]:
# reference spectra, set path to where you have them
ref_path = r'path/to/reference/file'
Ce4 = hs.load(ref_path+r"\Ce4_refnorm.hspy")
Ce3 = hs.load(ref_path+r"\Ce3_refnorm.hspy")

In [ ]:
# variables for saving and loading
name = r'Figure_74pA_InSitu_4'
out_dir = r'\manual'

path = r'\HighFlux\74pA_InSitu_(4)_'
sum_eels_path = path + "HL_stack_sumall.hspy"
proc_path_base = r'\HighFlux\74pA_InSitu_(4)_HL_stack_'
proc_path = proc_path_base+'sumall_'

In [ ]:
# load data - this may take A WHILE!
adf = hs.load(path+r"ADF_stack.hspy")
ll = hs.load(path+r"LL_stack.hspy")
hl = hs.load(path+r"HL_stack.hspy")
s = hs.load(sum_eels_path)
print(s.data.shape)# check the data shape shows a spectrum image (3 dimensions - x,y,energy)

In [ ]:
# load MLLS fits
eelsCe3glob = glob.glob(proc_path + r"bin2_ce3_map_mlls.npy")
ce3_path = eelsCe3glob[0]
eelsCe3MLLS = np.load(ce3_path) # _cleaned for session 1 /3
try:
    eelsCe4glob = glob.glob(proc_path + r"bin2_ce4_map_mlls.npy")
    ce4_path = eelsCe4glob[0]
    eelsCe4MLLS = np.load(ce4_path)
except (FileNotFoundError, OSError, IndexError):
    # if file doesn't exist, calculate from Ce3+ fits - given both are fitted together this is valid!
    eelsCe4MLLS = 1.0 - eelsCe3MLLS
dataglob = glob.glob(proc_path + r"postRL.npz")
datapath = dataglob[0]
data = np.load(datapath)

In [ ]:
# Access the arrays to check RL plots etc
print("Available arrays:", data.files)
eels_data = data['data']  # Shape: (height, width, energy_channels)
energy_axis = data['energy']  # Shape: (energy_channels,)

# processing image to check spectra
summed_spectrum = np.sum(eels_data, axis=(0, 1))
summed_spectrum = remove_spectral_spikes(summed_spectrum, threshold_sigma=2.0, window=9)
summed_spectrum_clean = np.clip(summed_spectrum, 0.0, None)
smoothed_spectrum = uniform_filter1d(summed_spectrum, size=3)
summed_spectrum = gaussian_filter1d(smoothed_spectrum, sigma=1)

print(f"EELS data shape: {eels_data.shape}")
print(f"Energy range: {energy_axis.min():.1f} - {energy_axis.max():.1f} eV")
print(f"Energy channels: {len(energy_axis)}")

# plot the spectrum as a figure to check the Ce M4 and M5 peaks are well aligned
# if not consider checking the metadata offset values are correct, and consider running LL.align_zero_loss_peak(also_align=HL)
# or work from raw data instead of deconvoled

plt.figure(figsize=(10, 6))
plt.plot(energy_axis, summed_spectrum, 'k-', linewidth=2)
plt.xlabel('Energy Loss (eV)')
plt.ylabel('Total Intensity')
plt.title('Summed Post-RL EELS Spectrum')
plt.grid(True, alpha=0.3)
plt.axvline(x=883, color='r', linestyle='--', alpha=0.5, label='Ce M5')
plt.axvline(x=901, color='b', linestyle='--', alpha=0.5, label='Ce M4')
plt.legend()
plt.show()

In [ ]:
# set up post RL data as summed frames from analysis
sum_eels = hs.signals.Signal1D(eels_data, metadata = s.metadata.as_dictionary(), signal_type="EELS")
sum_eels.axes_manager[-1].name = "Energy Loss"
sum_eels.axes_manager[-1].units = "eV"
sum_eels.axes_manager[-1].scale = float(energy_axis[1] - energy_axis[0])
sum_eels.axes_manager[-1].offset = float(energy_axis[0])
sum_eels.axes_manager[0].name = "x"
sum_eels.axes_manager[1].name = "y"
sum_eels.metadata.General.title = "Post-RL EELS"

In [ ]:
# rebinning to improve signal - optional!
sum_eels.rebin(new_shape=(sum_eels.data.shape[0]//2,sum_eels.data.shape[1]//2,sum_eels.data.shape[2]))
#sum_eels.plot()

In [ ]:
target_shape=(100, 100)
# Ce3 map
if eelsCe3MLLS.shape != target_shape:
    eelsCe3MLLS = eelsCe3MLLS[:target_shape[0], :target_shape[1]]
# Ce4 map
if eelsCe4MLLS.shape != target_shape:
    eelsCe4MLLS = eelsCe4MLLS[:target_shape[0], :target_shape[1]]

In [ ]:
# process ADF in same manner
adfsum = adf.sum()
adf_rebin = adfsum.rebin(new_shape=((adfsum.data.shape[0]//2),(adfsum.data.shape[1]//2)))
adf_rebin.axes_manager[0].scale = 0.3
adf_rebin.axes_manager[1].scale = 0.3
adf_rebin.data = adf_rebin.data[:target_shape[0], :target_shape[1]]
adf_data = adf_rebin.data
#adf_rebin.plot()

# the adf images are typically acquired as 400x400 for 200x200 SI
# therefore pixel size should be set as 0.075 if not binned, and when binned to 100x100, 0.3
adfsum.axes_manager[0].scale = 0.075
adfsum.axes_manager[1].scale = 0.075
adf.axes_manager[0].scale = 0.075
adf.axes_manager[1].scale = 0.075
#adfsum.plot()

In [ ]:
# Pixel size (nm) used for scalebars and converting px to nm
pixel_size = adf_rebin.axes_manager[0].scale # this will be 0.15 not binned, 0.3 for bin 2.
border_clearance_px = 3

## Main Multiframe DD EELS masking and processing

In [ ]:
# load distance binning statistics from CSV if available from earlier manual tuning
params_csv = f"{out_dir}/{name}_distance_binning_parameters.csv"
params_df = pd.read_csv(params_csv)
params = dict(zip(params_df['parameter'], params_df['value']))
percentile = params['mask_percentile']

In [ ]:
# Normalise & denoise adf
adf_float = adf_data.astype(float)
# uses li as a starting point, typically handles okay as a start
li_thresh = threshold_li(adf_float)
#Adjust percentile if mask is too aggressive/conservative, see plots
#percentile = .45
adf_mask = adf_float > li_thresh*percentile

labeled_mask, num_features = ndi.label(adf_mask)  
sizes = ndi.sum(adf_mask, labeled_mask, range(1, num_features+1))  
largest_label = np.argmax(sizes) + 1  
largest_mask = labeled_mask == largest_label

distance_map, mask_e, mask_b, tval = create_distance_masks(adf_data, existing_mask=largest_mask)
fig, axes = plt.subplots(1,3, figsize=(15,5))

axes[0].imshow(adf_data, cmap='gray'); axes[0].set_title('ADF'); axes[1].axis('off')
axes[1].imshow(adf_data, cmap='gray')
axes[1].imshow(np.ma.masked_where(~largest_mask, largest_mask), cmap='autumn', alpha=0.2)
axes[1].set_title(f'ADF + Li * {percentile} Mask Overlay'); axes[0].axis('off')

distance_map_nm = distance_map * pixel_size
distance_map_nm = np.ma.masked_invalid(distance_map_nm)# hide inf/NaN
im = axes[2].imshow(distance_map_nm.copy(), cmap='viridis')
axes[2].set_title('Distance Map (nm)'); axes[2].axis('off')
plt.show()
out_fig = os.path.join(out_dir, f"{name}_adf_mask.png")
if SAVE_FIGURES:
    save_figure(fig, out_fig, tight=False, dpi=1000)
    print('saved')
else:
    print('not saved') # check here

In [ ]:
# computes Ce3/(Ce3+Ce4) fraction vs distance with per-pixel standard deviation.
# Uses the distance_map computed earlier and valid_mask excluding border clearance
h, w = adf_mask.shape
yy, xx = np.indices((h, w))
dist_to_border = np.minimum.reduce([yy, h - 1 - yy, xx, w - 1 - xx])
valid_mask = adf_mask.copy()
valid_mask[dist_to_border < border_clearance_px] = False

max_distance_pixels = 100
bin_width_pixels = 1
distance_bins = np.arange(0, max_distance_pixels + bin_width_pixels, bin_width_pixels)
distance_centers = distance_bins[:-1] + bin_width_pixels/2

ce3_frac = []
ce3_frac_std = [] 
ce3_frac_sem = []
counts = []
min_pixels_threshold = 5  # Minimum pixels per bin to compute, <5 pixels insufficient for robust statistics so shoudlnt be plotted

for i in range(len(distance_bins)-1):
    b0, b1 = distance_bins[i], distance_bins[i+1]
    bin_mask = (distance_map >= b0) & (distance_map < b1) & valid_mask
    cnt = np.sum(bin_mask)
    counts.append(cnt)
    
    if cnt < min_pixels_threshold:
        ce3_frac.append(np.nan)
        ce3_frac_std.append(np.nan)
        ce3_frac_sem.append(np.nan)
        continue
    
    c3 = eelsCe3MLLS[bin_mask]
    c4 = eelsCe4MLLS[bin_mask]
    
    # Filter out zeros to avoid division issues
    valid = (c3 > 0) | (c4 > 0)
    if np.sum(valid) < min_pixels_threshold:
        ce3_frac.append(np.nan)
        ce3_frac_std.append(np.nan)
        ce3_frac_sem.append(np.nan)
        continue
    
    c3 = c3[valid]
    c4 = c4[valid]
    
    pixel_fractions = c3 / (c3 + c4)
    mean_frac = np.mean(pixel_fractions)
    std_frac = np.std(pixel_fractions, ddof=1)  # ddof=1 for sample SD
    sem_frac = std_frac / np.sqrt(len(pixel_fractions))  
    
    ce3_frac.append(mean_frac)
    ce3_frac_std.append(std_frac)
    ce3_frac_sem.append(sem_frac)

In [ ]:
# Save distance binning statistics to CSV
params_csv = f"{out_dir}/{name}_distance_binning_parameters.csv"
params_df = pd.DataFrame({
    'parameter': ['mask_percentile', 'bin_width_pixels', 'min_pixels_threshold', 'border_clearance_px'],
    'value': [percentile, bin_width_pixels, min_pixels_threshold, border_clearance_px]
})
params_df.to_csv(params_csv, index=False)

In [ ]:
# Plot MLLS maps, use matplotlib mathtext for superscripts in labels
mpl.rcParams['mathtext.default'] = 'regular'

# Inputs/checks
if 'eelsCe3MLLS' not in globals() or 'eelsCe4MLLS' not in globals():
    print('MLLS maps not found in memory. Load them and re-run this cell.')
else:
    ce3 = np.nan_to_num(eelsCe3MLLS, nan=0.0, posinf=0.0, neginf=0.0).astype(float)
    ce4 = np.nan_to_num(eelsCe4MLLS, nan=0.0, posinf=0.0, neginf=0.0).astype(float)
    assert ce3.shape == ce4.shape == largest_mask.shape, 'Shape mismatch between maps and mask'

    # Small smoothing for display
    smoothing_sigma = 0.8
    ce3_disp = gaussian_filter(ce3, sigma=smoothing_sigma) if smoothing_sigma>0 else ce3.copy()
    ce4_disp = gaussian_filter(ce4, sigma=smoothing_sigma) if smoothing_sigma>0 else ce4.copy()

    # Mask background
    ce3_masked = np.ma.masked_where(~largest_mask, ce3_disp)
    ce4_masked = np.ma.masked_where(~largest_mask, ce4_disp)

    # Build combined MLLS map 
    eps = 1e-12
    denom = ce3_disp + ce4_disp + eps
    frac_ce4 = ce4_disp / denom
    mlls_map = frac_ce4  
    mlls_masked = np.ma.masked_where(~largest_mask, mlls_map)

    # Colormap for combined map
    cmap_mlls = mpl.cm.get_cmap('plasma_r')
    cmap_mlls.set_bad('black')
    # Robust vmax/vmin for Ce maps
    def robust_vmax(arr, pct=99.0):
        vals = arr[arr>0]
        return float(np.percentile(vals, pct)) if vals.size>0 else float(np.max(arr))

    vmax3 = robust_vmax(ce3_disp, 99.0)
    vmax4 = robust_vmax(ce4_disp, 99.0)

In [ ]:
# exact 1 nm edge region and sum EELS within that mask
edge_distance_nm = 1.0
edge_threshold_px = edge_distance_nm / pixel_size
area_mask = (distance_map >= 0) & (distance_map <= edge_threshold_px) & largest_mask # or adf_mask
area_label = f'edge {edge_distance_nm:.2f} nm (<= {edge_threshold_px:.3f} px)'
centre_mask = (distance_map > edge_threshold_px) & largest_mask
print('Area selected for summing:', area_label, '->', int(np.sum(area_mask)), 'pixels')

# Load summed EELS 
s = sum_eels 
s = s.rebin(new_shape=(100,100,s.data.shape[2]))
data = np.asarray(s.data)
energy = s.axes_manager.signal_axes[0].axis
arr = np.asarray(data)

masked_data_edge = s.data * area_mask[:, :, np.newaxis]
masked_data_centre = s.data * centre_mask[:, :, np.newaxis]

cent_masked = hs.signals.Signal1D(masked_data_centre, metadata = s.metadata.as_dictionary(), signal_type="EELS")
edge_masked = hs.signals.Signal1D(masked_data_edge, metadata = s.metadata.as_dictionary(), signal_type="EELS")
cent_masked.axes_manager[2].scale = sum_eels.axes_manager[2].scale 
cent_masked.axes_manager[2].offset = sum_eels.axes_manager[2].offset 
cent_masked.axes_manager[2].units = "eV"
edge_masked.axes_manager[2].scale = sum_eels.axes_manager[2].scale
edge_masked.axes_manager[2].offset = sum_eels.axes_manager[2].offset 
edge_masked.axes_manager[2].units = "eV"

summed_signal1 = cent_masked.sum(axis=(0, 1)) 
summed_signal2 = edge_masked.sum(axis=(0, 1)) 

summed_signal1.axes_manager.signal_axes[0].name = "Energy Loss"
summed_signal1.axes_manager.signal_axes[0].name_in_label = "Energy Loss"
summed_signal1.axes_manager.signal_axes[0].units = "eV"

summed_signal2.axes_manager.signal_axes[0].name = "Energy Loss"
summed_signal2.axes_manager.signal_axes[0].name_in_label = "Energy Loss"
summed_signal2.axes_manager.signal_axes[0].units = "eV"
# crop to ROI, adjust to your liking.
s1 = summed_signal1.isig[3143:3417]
s2 = summed_signal2.isig[3143:3417]
energy = energy[3143:3417]
#s1 = summed_signal1
#s2 = summed_signal2

# clean up spectral artefacts
ceCentre_clean = remove_spectral_spikes(s1.data, threshold_sigma=1.5, window=5)
ceEdge_clean = remove_spectral_spikes(s2.data, threshold_sigma=1.5, window=5)

ceEdgecleanspec = normalise_spectra(ceEdge_clean)
ceCentrecleanspec = normalise_spectra(ceCentre_clean)

#plot to confirm
fig, ax = plt.subplots(figsize=(6,4))
ax.plot(s1.axes_manager[0].axis, ceCentrecleanspec, label="Centre (>1 nm from surface)", color = 'blue', alpha=0.8)
ax.plot(s2.axes_manager[0].axis, ceEdgecleanspec, label="Edge (<1 nm from surface)", color = 'red', alpha=0.8)
ax.set_xlabel('Energy Loss (eV)')
ax.set_ylabel('Normalised Intensity (a.u.)')
ax.set_xlim(879,911)
ax.set_ylim(-0.1,1.3)
ax.set_yticks([])
ax.grid(False)
fig.legend(
    loc='upper right',
    bbox_to_anchor=(0.90, .88),
    ncol=1,
    fontsize=9
)
plt.show()

# Save (clean filename to avoid issues with parentheses)
out_fig = os.path.join(out_dir, f"{name}_summed_region_spectrum.png")
out_fig = out_fig.replace('(', '_').replace(')', '_')
if SAVE_FIGURES:
    save_figure(fig, out_fig, tight=False, dpi=1000)
    print('saved')
plt.show()

In [ ]:
# main plot for bare CeO2 EELS data
fig = plt.figure(figsize=(13, 8))
fntsize = 14
lblsize = 18

# Top row tight spacing for maps, Bottom row more spacing for axes
gs = gridspec.GridSpec(2, 1, figure=fig, height_ratios=[1.5, 1.1], hspace=0.1, 
                       left=0.05, right=0.98, top=0.96, bottom=0.08)

# Top row sub-grid 3 maps with minimal gaps, bottom row sub-grid 2 plots with more space for axes
gs_top = gridspec.GridSpecFromSubplotSpec(1, 3, subplot_spec=gs[0], width_ratios=[1.4, 1.4, 1.5], wspace=0.04)
gs_bottom = gridspec.GridSpecFromSubplotSpec(1, 2, subplot_spec=gs[1], 
                                             width_ratios=[1, 1], wspace=0.2)

ax_adf = fig.add_subplot(gs_top[0])
ax_ce3 = fig.add_subplot(gs_top[1])
ax_combined = fig.add_subplot(gs_top[2])

ax_spec = fig.add_subplot(gs_bottom[0])
ax_thickness = fig.add_subplot(gs_bottom[1])

ax_adf.imshow(adf.data[1], cmap='gray')
adftxt = ax_adf.text(0.02, 0.98, '(a)', transform=ax_adf.transAxes, fontsize=lblsize, 
            fontweight='bold', va='top', ha='left', color='white')
adftxt.set_path_effects([pe.withStroke(linewidth=2, foreground='black')])#add if needed

lw = 2 if adf.axes_manager[0].scale == 0.15 else 4 # some adf images are at 400x400, some at 200x200. this handles thickness for either
ax_adf.axis('off')
add_scalebar(ax_adf, pixel_size_nm=adf.axes_manager[0].scale, scale_length_nm=5, 
             location='lower left', color='white', fontsize=fntsize, linewidth=lw)

ax_ce3.set_facecolor('black')
cmap_ce3 = mpl.colors.LinearSegmentedColormap.from_list('ce3', [(0,0,0),(1,1,0)])
cmap_ce3.set_bad('black')
im_ce3 = ax_ce3.imshow(ce3_masked, cmap=cmap_ce3, vmin=0, vmax=vmax3, interpolation='nearest')
ax_ce3.text(0.02, 0.98, '(b)', transform=ax_ce3.transAxes, fontsize=lblsize, 
            fontweight='bold', va='top', ha='left', color='white')
ax_ce3.text(0.80, 0.90, r'Ce$^{3+}$', transform=ax_ce3.transAxes, 
            color='yellow', fontsize=18, fontweight='bold', path_effects=[pe.withStroke(linewidth=2, foreground='black')])
ax_ce3.axis('off')
add_scalebar(ax_ce3, pixel_size_nm=pixel_size, scale_length_nm=5, 
             location='lower left', color='white', fontsize=fntsize, linewidth=1)

ax_combined.set_facecolor('black')
im_combined = ax_combined.imshow(mlls_masked, cmap=cmap_mlls, vmin=0, vmax=1.0, interpolation='nearest')
ax_combined.text(00.02, 0.98, '(c)', transform=ax_combined.transAxes, fontsize=lblsize, 
                 fontweight='bold', va='top', ha='left', color='white')
ax_combined.axis('off')
cbar = plt.colorbar(im_combined, ax=ax_combined, fraction = 0.04645, pad=0.02)
tick_positions = [0, 0.25, 0.5, 0.75, 1.0]
tick_labels = ['100:0', '75:25', '50:50', '25:75', '0:100']
cbar.set_ticks(tick_positions)
cbar.set_ticklabels(tick_labels)
cbar.set_label('Ce$^{3+}$ : Ce$^{4+}$ ratio (%)', fontsize=fntsize)
cbar.ax.tick_params(axis='both', labelsize=fntsize)

add_scalebar(ax_combined, pixel_size_nm=0.3, scale_length_nm=5, 
             location='lower left', color='white', fontsize=fntsize, linewidth=1)

# Prevent white corners on maps
for ax, data in [(ax_adf, adf.data[2]), (ax_ce3, ce3_masked), (ax_combined, mlls_masked)]:
    h, w = data.shape[:2]
    ax.set_xlim(-0.5, w - 0.5)
    ax.set_ylim(h - 0.5, -0.5)
    
# Plot experimental spectrum
ax_spec.plot(s1.axes_manager[0].axis, ceCentrecleanspec, label="Bulk (>1 nm)", 
             color='blue', alpha=0.8, linewidth=1.5)
ax_spec.plot(s2.axes_manager[0].axis, ceEdgecleanspec, label="Edge (<1 nm)", 
             color='red', alpha=0.8, linewidth=1.5)
ax_spec.set_xlabel('Energy Loss (eV)', fontsize=fntsize)
ax_spec.set_ylabel('Normalised Intensity (a.u.)', fontsize=fntsize)
ax_spec.set_xlim(879, 912)
ax_spec.set_ylim(-0.05, 1.15)
ax_spec.legend(loc='upper right', fontsize=9)
ax_spec.grid(True, alpha=0.3)
ax_spec.text(-0.14, 1.08, '(d)', transform=ax_spec.transAxes, fontsize=lblsize, 
             fontweight='bold', va='top', ha='left', color='black')
ax_spec.tick_params(axis='both', labelsize=fntsize)

# Plot thickness/distance curve with error bars
# Only plot points with sufficient pixels for reliable statistics
valid_points = ~np.isnan(ce3_frac)
distances_nm = distance_centers[valid_points] * pixel_size
means = np.array(ce3_frac)[valid_points]
stds = np.array(ce3_frac_std)[valid_points]
sems = np.array(ce3_frac_sem)[valid_points]

ax_thickness.errorbar(distances_nm, means, yerr=sems, 
                      marker='o', linestyle='-', color='darkblue', 
                      linewidth=1.5, markersize=3, elinewidth=1.0, 
                      capsize=3, capthick=0.8, alpha=0.8, label='Mean \u00b1 SEM')
ax_thickness.set_xlabel('Distance from surface (nm)', fontsize=fntsize)
ax_thickness.set_ylabel(r'Ce$^{3+}$/(Ce$^{3+}$ + Ce$^{4+}$)', fontsize=fntsize)
ax_thickness.set_xlim(0, int(distance_map_nm.max()))
ax_thickness.grid(True, alpha=0.3)
ax_thickness.text(-0.17, 1.08, '(e)', transform=ax_thickness.transAxes, fontsize=lblsize, 
                  fontweight='bold', va='top', ha='left', color='black')
ax_thickness.tick_params(axis='both', labelsize=fntsize)

for ax in (ax_adf, ax_ce3, ax_combined):# shift top row left by 0.03 to align with graphs etc given cbar labels
    pos = ax.get_position() 
    new_pos = [pos.x0 - 0.05, pos.y0, pos.width, pos.height]  
    ax.set_position(new_pos)

axcomb = ax_combined.get_position() 
cax = cbar.ax
pos = cax.get_position()
new_pos = [pos.x0 - 0.055, pos.y0, pos.width, pos.height]
cax.set_position(new_pos)

out_fig = os.path.join(out_dir, f"{name}_mosaic_figure_v2_sem.png")
out_fig = out_fig.replace('(', '_').replace(')', '_')
if SAVE_FIGURES:
    save_figure(fig, out_fig, tight=True, dpi=1000, pad_inches=0.2)
    print('saved')
plt.show()

## Fluence curve calculation
This section analyses the changes in oxidation state at the edge of the nanoparticles frame by frame, to track if any reduction may have occured during mapping.

In [ ]:
hl_rebin = hl.rebin(new_shape=(100, 100, hl.data.shape[0], hl.data.shape[3])) # bin 2 before to match adf

In [ ]:
hl_rebin.axes_manager[0].scale = 0.3
hl_rebin.axes_manager[1].scale = 0.3
hl_rebin.axes_manager

In [ ]:
energy_axis = hl_rebin.axes_manager[3].axis

In [ ]:
# Edge-region fluence analysis with single global M5 shift (sum-all)
def calculate_mlls_residual(spectrum, c3, c4, ce3_ref, ce4_ref):
    """Calculate MLLS fit residual and RMS error."""
    fitted = c3 * ce3_ref + c4 * ce4_ref
    residual = spectrum - fitted
    rms_error = np.sqrt(np.mean(residual**2))
    
    ss_res = np.sum(residual**2)
    ss_tot = np.sum((spectrum - np.mean(spectrum))**2)
    r_squared = 1 - (ss_res / ss_tot) if ss_tot > 0 else 0
    
    return rms_error, r_squared

# Parameters set up
edge_thickness_nm = 1.0
pixel_size_nm = float(pixel_size)  # already defined earlier, but if running this in isolation redefine!
edge_px = edge_thickness_nm / pixel_size_nm
edge_mask = (distance_map >= 0) & (distance_map <= edge_px) & adf_mask
edata = hl_rebin.data  # shape: (frames, y, x, E)
num_frames = edata.shape[0]

# Reference energy axis from reference spectra
ce3_energy = Ce3.axes_manager[0].axis
ce4_energy = Ce4.axes_manager[0].axis
ce3_ref = Ce3.data.astype(float).copy()
ce4_ref = Ce4.data.astype(float).copy()

# Target analysis energy window based on ref shape
ref_energy_offset = 870.0
ref_energy_high = 913.95
ref_scale = float(np.abs(ce3_energy[1] - ce3_energy[0]))
ref_energy_axis = ce3_energy[(ce3_energy >= ref_energy_offset) & (ce3_energy <= ref_energy_high)]
if ref_energy_axis.size < 10:
    raise ValueError("Not enough reference energy channels in selected window.")

sum_all_cube = cumulative_eels(num_frames, hl_rebin).data
sum_all_spec_norm = extract_edge_spectrum(sum_all_cube, edge_mask, energy_axis, ref_energy_offset, ref_energy_high, ref_energy_axis)
global_shift_ev = 0.0
if sum_all_spec_norm is not None:
    global_shift_ev = compute_m5_shift(sum_all_spec_norm, ref_energy_axis, ce4_ref, ce4_energy)
    if global_shift_ev != 0.0:
        ce3_ref_shifted = np.interp(ref_energy_axis, ce3_energy + global_shift_ev, ce3_ref)
        ce4_ref_shifted = np.interp(ref_energy_axis, ce4_energy + global_shift_ev, ce4_ref)
        ce3_ref = ce3_ref_shifted / max(ce3_ref_shifted.max(), 1e-12)
        ce4_ref = ce4_ref_shifted / max(ce4_ref_shifted.max(), 1e-12)
    else:
        ce3_ref = ce3_ref / max(ce3_ref.max(), 1e-12)
        ce4_ref = ce4_ref / max(ce4_ref.max(), 1e-12)

print(f"Applied global M5 shift: {global_shift_ev:.3f} eV")

current_pA = float(name.split('pA')[0].split('_')[-1])
scale_to_18 = current_pA / 18.0

rows = []
for f in range(1, num_frames + 1):
    cum_cube = cumulative_eels(f, hl_rebin)
    spec_norm = extract_edge_spectrum(cum_cube.data, edge_mask, energy_axis, ref_energy_offset, ref_energy_high, ref_energy_axis)
    c3, c4, c3_frac, ratio = mlls_fit(spec_norm, ce3_ref, ce4_ref)
    rms_error, r_squared = calculate_mlls_residual(spec_norm, c3, c4, ce3_ref, ce4_ref)
    # RMS to estimate noise in the spectrum
    N = len(spec_norm)
    sigma2 = (rms_error**2) * N / (N - 2) 
    # Covariance of the fit coeffs
    X = np.vstack([ce3_ref, ce4_ref]).T
    XtX_inv = np.linalg.inv(X.T @ X)
    cov = sigma2 * XtX_inv
    # Propagate to Ce3+ fraction
    denom = (c3 + c4)**2
    J = np.array([c4 / denom, -c3 / denom])
    ce3_frac_err = np.sqrt(J.T @ cov @ J)
    
    rows.append({
        'frame': f,
        'ce3_coeff': c3,
        'ce4_coeff': c4,
        'ce3_fraction': c3_frac,
        'ce3_ce4_ratio': ratio,
        'ce3_fraction_err': ce3_frac_err, 
        'edge_pixels': int(edge_mask.sum()),
        'rms_error_sumall': rms_error,     
        'r_squared_sumall': r_squared,     
        'dose_raw': f * current_pA,
        'dose_18eq': f * scale_to_18,
        'global_shift_eV': global_shift_ev
    })

edge_df = pd.DataFrame(rows)

# Plot
fig = plt.figure(figsize=(8, 5))

plt.errorbar(
    edge_df['dose_18eq'],
    edge_df['ce3_fraction'],
    yerr=edge_df['ce3_fraction_err'],
    marker='o',
    linestyle='-',
    linewidth=2,
    capsize=4,
    alpha=0.8
)

plt.xlabel('Equivalent 18 pA Frames', fontweight='bold')
plt.ylabel('Ce$^{3+}$ Fraction (edge)', fontweight='bold')
plt.ylim(0.0, 0.8)
plt.grid(alpha=0.3, linestyle='--')
plt.tight_layout()

out_fig = os.path.join(out_dir, f"{name}_fluence_plot_err_sem.png")
if SAVE_FIGURES:
    save_figure(fig, out_fig, tight=False, dpi=1000)
plt.show()

In [ ]:
# Visualise edge region used in fluence analysis
edge_thickness_nm = 1.0
pixel_size_nm = float(pixel_size)  # nm per (rebinned) pixel
edge_px = edge_thickness_nm / pixel_size_nm

# Edge mask (rim) and centre mask (interior)
edge_mask = (distance_map >= 0) & (distance_map <= edge_px) & adf_mask
centre_mask = (distance_map > edge_px) & adf_mask

print(f"Pixel size (nm): {pixel_size_nm:.3f}")
print(f"Edge thickness target (nm): {edge_thickness_nm:.2f}")
print(f"Edge threshold (pixels): {edge_px:.2f}")
print(f"Edge pixels: {edge_mask.sum()}  | Centre pixels: {centre_mask.sum()}")

# Convert distance map to nm for display
distance_map_nm = distance_map * pixel_size_nm
distance_map_nm_masked = np.ma.masked_where(~adf_mask, distance_map_nm)

fig, axes = plt.subplots(1, 4, figsize=(14,4))

# ADF
axes[0].imshow(adf_data, cmap='gray')
axes[0].set_title('ADF')
axes[0].axis('off')

# Distance map (nm)
im1 = axes[1].imshow(distance_map_nm_masked, cmap='viridis')
axes[1].set_title('Distance to boundary (nm)')
axes[1].axis('off')
plt.colorbar(im1, ax=axes[1], fraction=0.046)

# Edge mask overlay
axes[2].imshow(adf_data, cmap='gray')
axes[2].imshow(np.ma.masked_where(~edge_mask, edge_mask), cmap='autumn', alpha=0.45, interpolation='nearest')
axes[2].set_title(f'Edge mask (≤ {edge_thickness_nm:.1f} nm)')
axes[2].axis('off')

# Centre mask overlay
axes[3].imshow(adf_data, cmap='gray')
axes[3].imshow(np.ma.masked_where(~centre_mask, centre_mask), cmap='winter', alpha=0.45, interpolation='nearest')
axes[3].set_title(f'Centre (> {edge_thickness_nm:.1f} nm)')
axes[3].axis('off')

plt.tight_layout()
out_fig = os.path.join(out_dir, f"{name}_edge_region_visualisation.png")
if SAVE_FIGURES:
    save_figure(fig, out_fig, tight=False, dpi=1000)
    print('save')
plt.show()

In [ ]:
csv_filename = f"{name}_fluence_plot_data_rms.csv"
csv_path = os.path.join(out_dir, csv_filename)
edge_df.to_csv(csv_path, index=False)
print(f'saved {csv_path}')

## Pd to Ce Oxidation State Correlation Analysis

This section loads Pd EDS data and analyses spatial correlation between Pd proximity and Ce oxidation state. 

In [ ]:
eds = hs.load(path+r"EDS_stack.hspy")

In [ ]:
eds.axes_manager
eds_sum = eds.sum(axis=(2))

# Rebin to match EELS/ADF if needed
if eds_sum.data.shape[1] != adf_mask.shape[1]:
    zoom_factors = ( eds_sum.data.shape[0]/ adf_mask.shape[0] , 
                   eds_sum.data.shape[1]/ adf_mask.shape[1],
                   1)
    eds_binned = eds_sum.rebin(scale=(zoom_factors))
    print(f"Rebinned EDS to match: {eds_binned.data.shape}")
else:
    eds_binned = eds_sum
eds_binned.add_elements(['Ce', 'Pd'])
# get all xray lines for Pd and Ce
eds_binned.add_lines(['Ce_La','Ce_Lb1','Pd_La','Pd_Lb1'])
bw = eds_binned.estimate_background_windows(line_width=[5.0, 2.0])
maps = eds_binned.get_lines_intensity(background_windows=bw, plot_result=False)

# you can also get Ce if wanted! but this focuses on Pd. run "maps" to see listed info on each within
pd_map = maps[2].data
pd_map += maps[3].data

pd_smooth = smooth_map(pd_map, method='gaussian', sigma=1)
pd_smooth[pd_smooth < 0] = 0

In [ ]:
fntsize = 16
lblsize = 20
# X-ray line energies (keV) - use eds_binned.get_lines_intensity() to check these!
ce_la = 4.84
ce_lb1 = 5.26
pd_la = 2.84
pd_lb1 = 2.99

fig = plt.figure(figsize=(14, 8))
gs = gridspec.GridSpec(1, 2, figure=fig, hspace=0.3, wspace=0.3,
                       left=0.08, right=0.96, top=0.92, bottom=0.1)

# Sum entire EDS map
overall_spectrum = eds_binned.sum(axis=(0, 1))

#  Ce + Pd ROI combined (2.2-5.5 keV)
ax2 = fig.add_subplot(gs[0, 0])
spectrum_ce_pd = overall_spectrum.isig[2.2:5.5]
ax2.plot(spectrum_ce_pd.axes_manager[0].axis, spectrum_ce_pd.data, 
         color='black', linewidth=2.5, alpha=0.9)
ax2.fill_between(spectrum_ce_pd.axes_manager[0].axis, spectrum_ce_pd.data, 
                 alpha=0.08, color='gray')
ax2.axvline(ce_la, color='red', linestyle='--', alpha=0.5, linewidth=1.5, label=r'Ce L$_{\alpha}$')
ax2.axvline(ce_lb1, color='darkred', linestyle='--', alpha=0.5, linewidth=1.5, label=r'Ce L$_{\beta_1}$')
ax2.axvline(pd_la, color='blue', linestyle='--', alpha=0.5, linewidth=1.5, label=r'Pd L$_{\alpha}$')
ax2.axvline(pd_lb1, color='darkblue', linestyle='--', alpha=0.5, linewidth=1.5, label=r'Pd L$_{\beta_1}$')
ax2.set_xlabel('Energy (keV)', fontsize=fntsize)
ax2.set_ylabel('Counts', fontsize=fntsize)
ax2.set_xlim(2.2,5.5)
ax2.legend(fontsize=fntsize, loc='upper center', framealpha=0.9)
ax2.text(0.02, 0.98, '(a)', transform=ax2.transAxes, fontsize=lblsize, 
            fontweight='bold', va='top', ha='left', color='black')

# Pd L-lines region only 
ax3 = fig.add_subplot(gs[0, 1])
spectrum_pd = overall_spectrum.isig[2.5:3.4]
ax3.plot(spectrum_pd.axes_manager[0].axis, spectrum_pd.data, 
         color='black', linewidth=2.5, alpha=0.9)
ax3.fill_between(spectrum_pd.axes_manager[0].axis, spectrum_pd.data, 
                 alpha=0.08, color='gray')
ax3.axvline(pd_la, color='blue', linestyle='--', alpha=0.6, linewidth=1.5, label=r'Pd L$_{\alpha}$')
ax3.axvline(pd_lb1, color='darkblue', linestyle='--', alpha=0.6, linewidth=1.5, label=r'Pd L$_{\beta_1}$')
ax3.set_xlabel('Energy (keV)', fontsize=fntsize)
ax3.set_ylabel('Counts', fontsize=fntsize)
ax3.text(0.02, 0.98, '(b)', transform=ax3.transAxes, fontsize=lblsize, 
            fontweight='bold', va='top', ha='left', color='black')
ax3.set_xlim(2.5,3.4)
ax3.legend(fontsize=fntsize, loc='upper right', framealpha=0.9)
for ax in fig.axes:
    ax.tick_params(axis='both', which='both', labelsize=fntsize)
    ax.set_ylim(ymin=0)
    ax.grid(True, alpha=0.3, linestyle='--')
plt.tight_layout()

# Save figure
out_fig = os.path.join(out_dir, f"{name}_EDS_spectra.png")
out_fig = out_fig.replace('(', '_').replace(')', '_')
if SAVE_FIGURES:
    save_figure(fig, out_fig, tight=True, dpi=1000, pad_inches=0.15)
    print(f"Saved EDS spectral comparison figure to: {out_fig}")

plt.show()

In [ ]:
# Visualise Pd map
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(adf_data, cmap='gray')
axes[0].set_title('ADF')
axes[0].axis('off')
im = axes[1].imshow(pd_smooth, cmap='hot')
axes[1].set_title('Pd EDS Map')
axes[1].axis('off')
plt.colorbar(im, ax=axes[1], fraction=0.046)
plt.tight_layout()
plt.show()

In [ ]:
# Resize pd_smooth to match Ce3+ map shape
pd_smooth_resized = resize(
    pd_smooth,
    ce3_masked.shape,
    order=1,  # linear interpolation
    preserve_range=True,
    anti_aliasing=True
)

pd_source = pd_smooth_resized > 0.8
distance_mask = (~pd_source) & largest_mask
distance_map_pd = distance_transform_edt(distance_mask)

# Parameters
pixel_size_nm = 0.3
bin_width = 1

# Compute 
max_distance = int(np.nanmax(distance_map_pd[largest_mask]))
bin_edges = np.linspace(0, max_distance, int(max_distance/bin_width) + 1)
bin_centerspd = bin_edges[:-1] + bin_width / 2

ce3_radialpd = []
ce3_radialpd_std = []
ce3_radialpd_sem = []
counts = []

for i in range(len(bin_edges)-1):
    b0, b1 = bin_edges[i], bin_edges[i+1]
    bin_mask = (distance_map_pd >= b0) & (distance_map_pd < b1) & largest_mask
    cnt = np.sum(bin_mask)
    counts.append(cnt)
    if cnt < 5:
        ce3_radialpd.append(np.nan)
        ce3_radialpd_std.append(np.nan)
        ce3_radialpd_sem.append(np.nan)
        continue
    ce3_vals = ce3_masked[bin_mask]
    valid = ce3_vals > 0
    n_valid = np.sum(valid)
    if np.sum(valid) < 5:
        ce3_radialpd.append(np.nan)
        ce3_radialpd_std.append(np.nan)
        ce3_radialpd_sem.append(np.nan)
        continue
    ce3_valid = ce3_vals[valid]
    ce3_radialpd.append(np.mean(ce3_valid))
    ce3_pd_std = np.std(ce3_valid, ddof=1)
    ce3_radialpd_std.append(ce3_pd_std)
    ce3_pd_sem = ce3_pd_std / np.sqrt(n_valid)
    ce3_radialpd_sem.append(np.mean(ce3_pd_sem))
    
ce3_radialpd = np.array(ce3_radialpd)
ce3_radialpd_std = np.array(ce3_radialpd_std)
ce3_radialpd_sem = np.array(ce3_radialpd_sem)

bin_centerspd_trim = bin_centerspd[1:]
ce3_radialpd_trim = ce3_radialpd[1:]
ce3_radialpd_std_trim = ce3_radialpd_std[1:]
ce3_radialpd_sem_trim = ce3_radialpd_sem[1:]

# Plot
fig, ax = plt.subplots(figsize=(6, 5))
ax.errorbar(bin_centerspd_trim * pixel_size_nm, ce3_radialpd_trim, yerr=ce3_radialpd_sem_trim,
            marker='o', linestyle='-', color='blue', elinewidth=1.0, capsize=3, 
            capthick=0.8, alpha=0.8, label='Mean \u00b1 SEM')
ax.set_xlabel('Distance from Pd (nm)', fontsize=14)
ax.set_ylabel('Ce$^{3+}$ fraction', fontsize=14)
ax.tick_params(axis='both', labelsize=14)
ax.set_xlim(0, max_distance * pixel_size_nm)
ax.grid(True, alpha=0.3)
ax.legend(loc='best', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Clean the distance map by clipping infinite values for plot to get accurate max nm
distance_map_nm_clean = np.nan_to_num(distance_map * pixel_size_nm, nan=0.0, posinf=0.0, neginf=0.0)
distance_map_nm_masked_clean = np.ma.masked_where(~adf_mask, distance_map_nm_clean)
plotpdmax = distance_map_nm_clean[adf_mask].max()
print(f"Max distance (nm): {plotpdmax:.2f}")
# mask for plotting - this area is removed in distance calculations anyways!
pd_masked = np.zeros_like(pd_smooth)
pd_masked[largest_mask] = pd_smooth[largest_mask]

In [ ]:
#final figure with everything plotted!
fig = plt.figure(figsize=(16, 8))
fntsize = 16
lblsize = 20

gs = gridspec.GridSpec(2, 1, figure=fig, height_ratios=[1.5, 1.1], hspace=0.15, 
                       left=0.05, right=0.98, top=0.96, bottom=0.08)

gs_top = gridspec.GridSpecFromSubplotSpec(1, 4, subplot_spec=gs[0],  width_ratios=[1.4, 1.4, 1.4, 1.5], wspace=0.04)
gs_bottom = gridspec.GridSpecFromSubplotSpec(1, 3, subplot_spec=gs[1],  width_ratios=[1, 1, 1], wspace=0.2)

ax_adf = fig.add_subplot(gs_top[0])
ax_ce3 = fig.add_subplot(gs_top[2])
ax_combined = fig.add_subplot(gs_top[3])
ax_eds = fig.add_subplot(gs_top[1])

ax_spec = fig.add_subplot(gs_bottom[0])
ax_thickness = fig.add_subplot(gs_bottom[1])
ax_pd = fig.add_subplot(gs_bottom[2])  

ax_adf.imshow(adf.data[1], cmap='gray')
adftxt = ax_adf.text(0.02, 0.98, '(a)', transform=ax_adf.transAxes, fontsize=lblsize, 
            fontweight='bold', va='top', ha='left', color='white')
adftxt.set_path_effects([pe.withStroke(linewidth=2, foreground='black')])
lw = 2 if adf.axes_manager[0].scale == 0.15 else 4
ax_adf.axis('off')
add_scalebar(ax_adf, pixel_size_nm=0.075, scale_length_nm=5, 
             location='lower left', color='white', fontsize=fntsize, linewidth=lw)

ax_ce3.set_facecolor('black')
im_ce3 = ax_ce3.imshow(ce3_masked, cmap=cmap_ce3, vmin=0, vmax=vmax3, interpolation='nearest')
ax_ce3.text(0.02, 0.98, '(c)', transform=ax_ce3.transAxes, fontsize=lblsize, 
            fontweight='bold', va='top', ha='left', color='white')
ax_ce3.text(0.80, 0.90, r'Ce$^{3+}$', transform=ax_ce3.transAxes, 
            color='yellow', fontsize=18, fontweight='bold', path_effects=[pe.withStroke(linewidth=2, foreground='black')])
ax_ce3.axis('off')
add_scalebar(ax_ce3, pixel_size_nm=0.3, scale_length_nm=5, 
             location='lower left', color='white', fontsize=fntsize, linewidth=1)

ax_combined.set_facecolor('black')
im_combined = ax_combined.imshow(mlls_masked, cmap=cmap_mlls, vmin=0, vmax=1.0, interpolation='nearest')
ax_combined.text(0.02, 0.98, '(d)', transform=ax_combined.transAxes, fontsize=lblsize, 
                 fontweight='bold', va='top', ha='left', color='white')
ax_combined.axis('off')
cbar = plt.colorbar(im_combined, ax=ax_combined, fraction = 0.04645, pad=0.02)
tick_positions = [0, 0.25, 0.5, 0.75, 1.0]
tick_labels = ['100:0', '75:25', '50:50', '25:75', '0:100']
cbar.set_ticks(tick_positions)
cbar.set_ticklabels(tick_labels)
cbar.set_label('Ce$^{3+}$ : Ce$^{4+}$ ratio (%)', fontsize=fntsize)
cbar.ax.tick_params(axis='both', labelsize=fntsize)
add_scalebar(ax_combined, pixel_size_nm=0.3, scale_length_nm=5, 
             location='lower left', color='white', fontsize=fntsize, linewidth=1)

ax_eds.imshow(pd_masked, cmap='hot', interpolation='nearest')
ax_eds.text(0.02, 0.98, '(b)', transform=ax_eds.transAxes, fontsize=lblsize,
            fontweight='bold', va='top', ha='left', color='white')
ax_eds.axis('off')
add_scalebar(ax_eds, pixel_size_nm=0.3, scale_length_nm=5, 
             location='lower left', color='white', fontsize=fntsize, linewidth=1)
ax_eds.text(0.97, 0.97, r'Pd', transform=ax_eds.transAxes, va='top', ha='right',
            color='orangered', fontsize=18, fontweight='bold', path_effects=[pe.withStroke(linewidth=3, foreground='black')])

for ax, data in [(ax_adf, adf.data[2]), (ax_ce3, ce3_masked), (ax_combined, mlls_masked), (ax_eds, pd_smooth)]:
    h, w = data.shape[:2]
    ax.set_xlim(-0.5, w - 0.5)
    ax.set_ylim(h - 0.5, -0.5)

ax_spec.plot(s1.axes_manager[0].axis, ceCentrecleanspec, label="Bulk (>1 nm)", color='blue', alpha=0.8, linewidth=1.5)
ax_spec.plot(s2.axes_manager[0].axis, ceEdgecleanspec, label="Edge (<1 nm)", color='red', alpha=0.8, linewidth=1.5)
ax_spec.set_xlabel('Energy Loss (eV)', fontsize=fntsize)
ax_spec.set_ylabel('Normalised Intensity (a.u.)', fontsize=fntsize)
ax_spec.set_xlim(879, 912)
ax_spec.set_ylim(-0.05, 1.25)
ax_spec.legend(loc='upper right', fontsize=9)
ax_spec.grid(True, alpha=0.3)
ax_spec.text(-0.18, 1.15, '(e)', transform=ax_spec.transAxes, fontsize=lblsize, fontweight='bold', va='top', ha='left', color='black')
ax_spec.tick_params(axis='both', labelsize=fntsize)

# Plot with error bars
valid_points = ~np.isnan(ce3_frac)
distances_nm = distance_centers[valid_points] * pixel_size
means = np.array(ce3_frac)[valid_points]
stds = np.array(ce3_frac_std)[valid_points]
sems = np.array(ce3_frac_sem)[valid_points]

ax_thickness.errorbar(distances_nm, means, yerr=sems, 
                      marker='o', linestyle='-', color='darkblue', linewidth=1.5, markersize=3,
                      elinewidth=1.0, capsize=3, capthick=0.8, alpha=0.8, label='Mean \u00b1 SEM')
ax_thickness.set_xlabel('Distance from surface (nm)', fontsize=fntsize)
ax_thickness.set_ylabel(r'Ce$^{3+}$/(Ce$^{3+}$ + Ce$^{4+}$)', fontsize=fntsize)
ax_thickness.set_xlim(0, int(plotpdmax))
ax_thickness.grid(True, alpha=0.3)
ax_thickness.text(-0.17, 1.14, '(f)', transform=ax_thickness.transAxes, fontsize=lblsize, 
                  fontweight='bold', va='top', ha='left', color='black')
ax_thickness.tick_params(axis='both', labelsize=fntsize)

# Plot with error bars if available
if 'ce3_radialpd_sem_trim' in locals():
    ax_pd.errorbar(bin_centerspd_trim * pixel_size_nm, ce3_radialpd_trim, yerr=ce3_radialpd_sem_trim,
                   marker='o', linestyle='-', color='blue', linewidth=1.5, markersize=3,
                   elinewidth=1.0, capsize=3, capthick=0.8, alpha=0.8, label='Mean \u00b1 SEM')
else:
    ax_pd.plot(bin_centerspd_trim * pixel_size_nm, ce3_radialpd_trim, 'o-', color='blue', label='Ce$^{3+}$ fraction')
ax_pd.set_xlabel('Distance from Pd (nm)', fontsize=fntsize)
ax_pd.set_ylabel('Ce$^{3+}$ fraction', fontsize=fntsize)
ax_pd.set_xlim(0, max_distance * pixel_size_nm)
ymax_rounded = np.ceil(np.nanmax(ce3_radialpd_trim) * 10) / 10
ax_pd.set_ylim(0, 0.25)#ymax_rounded)
ax_pd.grid(True, alpha=0.3)
ax_pd.text(-0.17, 1.14, '(g)', transform=ax_pd.transAxes, fontsize=lblsize,
                   fontweight='bold', va='top', ha='left', color='black')
ax_pd.tick_params(axis='both', labelsize=fntsize)

for ax in (ax_adf, ax_ce3, ax_eds, ax_combined):
    pos = ax.get_position() 
    new_pos = [pos.x0 - 0.05, pos.y0, pos.width, pos.height]  
    ax.set_position(new_pos)

axcomb = ax_combined.get_position() 
cax = cbar.ax
pos = cax.get_position()
new_pos = [pos.x0 - 0.05, pos.y0, pos.width, pos.height]
cax.set_position(new_pos)

# optional pd annotations for notable map correlations
pd_locations = []
for y, x in pd_locations:
    # On Pd map
    ax_eds.plot(x, y, marker='o', markersize=30, markeredgecolor='white', 
                markerfacecolor='none', lw=1.5) 
    # On Ce3+ map
    ax_ce3.plot(x, y, marker='o', markersize=30, markeredgecolor='white', 
                markerfacecolor='none', lw=1.5)
out_fig = os.path.join(out_dir+'Pd', f"{name}_mosaic_figure_Pd_sem.png")
out_fig = out_fig.replace('(', '_').replace(')', '_')
if SAVE_FIGURES:
    save_figure(fig, out_fig, tight=True, dpi=1000, pad_inches=0.2)
    print('saved')
plt.show()

In [ ]:
# save data involved in plot
os.makedirs(out_dir, exist_ok=True)

np.savez_compressed(os.path.join(out_dir, f'{name}_plot_data_Pd.npz'),
                    ce3_masked=ce3_masked,
                    pd_masked=pd_masked,
                    mlls_masked=mlls_masked,
                    adf_layer1=adf.data[1],
                    ce3_radialpd=ce3_radialpd,
                    bin_centerspd=bin_centerspd,
                    distance_centers=distance_centers,
                    ce3_frac=ce3_frac,
                    pd_smooth=pd_smooth)

## Comparing fluence plots, across flux and different samples

This section loads all fluence curves, as calculated above, to compare typical Ce$^{3+}$ fraction changes over time.

In [ ]:
# Root folder where manual fluence CSVs were saved - edit based on path!
root_dir = r"\ProcessingParams"

# Display names for samples in plots
sample_names = {
    'Bare_DecompCeria': 'CeO$_2$ (PD)',
    'Pd_DecompCeria': 'Pd/CeO$_2$ (PD)',   
    'Bare_FSPCeria': 'CeO$_2$ (FSP)',
}

# Colours per sample: lighter for 18 pA, darker for 74 pA
# any hex code will work
sample_colours = {
    'Bare_DecompCeria': {'18': '#fe7D6A', '74': '#082a54'},  # peach/navy
    'Pd_DecompCeria': {'18': '#a559aa', '74': '#e02b35'},    # purple/red
    'Bare_FSPCeria': {'18': '#59a89c', '74': '#BA7C00'},     # teal/yellow
}

# Beam current labels
current_labels = {
    18: r'1.25×10$^5$ e$^-$nm$^{-2}$s$^{-1}$',
    74: r'5.13×10$^5$ e$^-$nm$^{-2}$s$^{-1}$'
}

# Markers and linestyles
current_markers = {18: 'o', 74: 's'}
current_linestyles = {18: '-', 74: '--'}

# Dose calculation parameters
pixel_size_nm = 0.3
pixel_area_nm2 = pixel_size_nm ** 2
scan_pixels = 100 * 100
time_per_frame = {18: 12, 74: 12}

In [ ]:
# Load CSVs
pattern = "**/*_fluence_plot_data_rms.csv" 
records = []

for csv_path in glob.glob(os.path.join(root_dir, pattern), recursive=True):
    try:
        df = pd.read_csv(csv_path)
    except Exception:
        continue

    base = os.path.splitext(os.path.basename(csv_path))[0]
    name_part = base.replace("_fluence_plot_data_rms", "")
    m_cur = re.search(r'(\d+)\s*pA', name_part, re.IGNORECASE)
    current_pA = int(m_cur.group(1)) if m_cur else None
    
    parts = csv_path.split(os.sep)
    sample_id = parts[-2]
    m_stack = re.search(r'pA_(.*)', name_part, re.IGNORECASE)
    stack_name = m_stack.group(1) if m_stack else name_part

    if 'frame' not in df.columns or 'ce3_fraction' not in df.columns:
        continue

    df['sample_id'] = sample_id
    df['current_pA'] = current_pA
    df['stack_name'] = stack_name
    df['electron_dose'] = df['frame'].apply(lambda f: calculate_dose(current_pA, f, time_per_frame.get(current_pA, 42.123), scan_pixels, pixel_area_nm2))
    df['dose_18eq'] = df.get('dose_18eq', df['frame'] * (current_pA / 18.0 if current_pA else np.nan))
    df['dose_raw'] = df.get('dose_raw', df['frame'] * current_pA)
    records.append(df)

if not records:
    raise ValueError("No fluence CSVs found.")

all_df = pd.concat(records, ignore_index=True)
all_df = all_df[all_df['sample_id'].isin(sample_names.keys())]

avg_frame = (
    all_df
    .groupby(['sample_id','current_pA','frame'], as_index=False)
    .agg(
        ce3_mean=('ce3_fraction','mean'),
        ce3_std=('ce3_fraction','std'),
        n=('ce3_fraction','count'),
        ce3_fraction_err_multi =('ce3_fraction_err', lambda x: np.sqrt(np.sum(x**2)) / len(x)),
        dose_18eq=('dose_18eq','mean'),
        electron_dose=('electron_dose','mean')
    )
)
avg_frame = avg_frame[avg_frame.apply(
    lambda row: all_df[
        (all_df['sample_id'] == row['sample_id']) &
        (all_df['current_pA'] == row['current_pA']) &
        (all_df['frame'] == row['frame'])
    ]['stack_name'].nunique() > 1,
    axis=1
)]

# Helper to compute ylim for plot with minimum range
def compute_ylim(data_arrays, yerr_arrays, min_range=0.1, margin_frac=0.05):
    y_all = np.concatenate(data_arrays)
    yerr_all = np.concatenate(yerr_arrays)
    lower = y_all - yerr_all
    upper = y_all + yerr_all
    data_min = np.nanmin(lower)
    data_max = np.nanmax(upper)
    span = max(data_max - data_min, min_range)
    centre = 0.5 * (data_max + data_min)
    margin = max(margin_frac * span, 0.01)
    return centre - span/2 - margin, centre + span/2 + margin

# One figure per sample
samples = avg_frame['sample_id'].unique()
for sample in samples:
    sub = avg_frame[avg_frame.sample_id == sample]
    for current in sorted(sub.current_pA.unique()):
        csvs_for_current = all_df[
            (all_df['sample_id'] == sample) & 
            (all_df['current_pA'] == current)
        ]['stack_name'].unique()
        print(f"Sample {sample}, current {current} pA will plot from CSVs: {list(csvs_for_current)}")
        for csv_name in csvs_for_current:
            n_rows = len(all_df[
                (all_df['sample_id'] == sample) &
                (all_df['current_pA'] == current) &
                (all_df['stack_name'] == csv_name)
            ])

    fig, ax = plt.subplots(figsize=(8, 5))
    data_arrays = []
    yerr_arrays = []
    for current in sorted(sub.current_pA.unique()):
        cur_df = sub[sub.current_pA == current].sort_values('electron_dose')
        data_arrays.append(cur_df['ce3_mean'].values)
        yerr_arrays.append(cur_df['ce3_fraction_err_multi'].values)
        label = current_labels.get(current, f'{current} pA')
        ax.errorbar(cur_df['electron_dose'], cur_df['ce3_mean'], yerr=cur_df['ce3_fraction_err_multi'],
                marker=current_markers[current], linestyle=current_linestyles[current],
                linewidth=2, markersize=6, color=sample_colours[sample][str(current)], alpha=0.85,
                label=label, capsize=3)
    
    ax.set_xlabel('Electron Fluence (e$^-$/nm$^2$)', fontsize=14, fontweight='bold')
    ax.set_ylabel('Ce$^{3+}$ Fraction (edge)', fontsize=14, fontweight='bold')
    ax.set_ylim(compute_ylim(data_arrays, yerr_arrays, min_range=0.1))
    ax.set_title(f'{sample_names[sample]}', fontsize=14, fontweight='bold', pad=20)
    ax.grid(alpha=0.3, linestyle=':')
    ax.tick_params(axis='both', labelsize=14)
    ax.legend(fontsize=14, loc='best')
    ax.ticklabel_format(style='scientific', axis='x', scilimits=(0,0))
    plt.tight_layout()
    out_fig = os.path.join(root_dir, f"{sample}_fluence_average_electron_dose_v3_pixdwell.png")
    if SAVE_FIGURES:
        plt.savefig(out_fig, dpi=1000, bbox_inches='tight')
    plt.show()

# Save aggregated CSV
avg_frame.to_csv(os.path.join(root_dir, "all_fluence_averages_electron_dose.csv"), index=False)

In [ ]:
samples = avg_frame['sample_id'].unique()
for sample in samples:
    if sample == 'Bare_FSPCeria':
        sub = avg_frame[avg_frame.sample_id == sample]
        for current in sorted(sub.current_pA.unique()):
            csvs_for_current = all_df[
                (all_df['sample_id'] == sample) & 
                (all_df['current_pA'] == current)
            ]['stack_name'].unique()
            print(f"Sample {sample}, current {current} pA will plot from CSVs: {list(csvs_for_current)}")
            for csv_name in csvs_for_current:
                n_rows = len(all_df[
                    (all_df['sample_id'] == sample) &
                    (all_df['current_pA'] == current) &
                    (all_df['stack_name'] == csv_name)
                ])    
        fig, ax = plt.subplots(figsize=(8, 5))
        data_arrays = []
        yerr_arrays = []
        for current in sorted(sub.current_pA.unique()):
            cur_df = sub[sub.current_pA == current].sort_values('electron_dose')
            data_arrays.append(cur_df['ce3_mean'].values)
            yerr_arrays.append(cur_df['ce3_fraction_err_multi'].values)
            label = current_labels.get(current, f'{current} pA')
            ax.errorbar(cur_df['electron_dose'], cur_df['ce3_mean'], yerr=cur_df['ce3_fraction_err_multi'],
                    marker=current_markers[current], linestyle=current_linestyles[current],
                    linewidth=2, markersize=6, color=sample_colours[sample][str(current)], alpha=0.85,
                    label=label, capsize=3)
        
        ax.set_xlabel('Electron Fluence (e$^-$/nm$^2$)', fontsize=14, fontweight='bold')
        ax.set_ylabel('Ce$^{3+}$ Fraction (edge)', fontsize=14, fontweight='bold')
        ymin, ymax = compute_ylim(data_arrays, yerr_arrays, min_range=0.1)
        ax.set_ylim(ymin-0.05, ymax)
        ax.set_title(f'{sample_names[sample]}', fontsize=14, fontweight='bold', pad=20)
        ax.grid(alpha=0.3, linestyle=':')
        ax.tick_params(axis='both', labelsize=14)
        ax.legend(fontsize=14, loc='lower right')
        ax.ticklabel_format(style='scientific', axis='x', scilimits=(0,0))
        plt.tight_layout()
        out_fig = os.path.join(root_dir, f"{sample}_fluence_average_electron_dose_v3_pixdwell.png")
        if SAVE_FIGURES:
            plt.savefig(out_fig, dpi=1000, bbox_inches='tight')
        plt.show()

In [ ]:
# to compare any 2 specific datasets, and get their stats.
def load_fluence_data(sample_folder, current_pA, stack_identifier):
    """Load specific fluence CSV based on sample folder, current, and stack name"""    
    pattern = os.path.join(root_dir, sample_folder, 'manual', 
                          f"*{current_pA}pA*{stack_identifier}*fluence_plot_data_rms.csv")
    matches = glob.glob(pattern)
    
    if not matches:
        print(f"No CSV found: {sample_folder} | {current_pA}pA | {stack_identifier}")
        return None
    
    if len(matches) > 1:
        print(f"Multiple CSVs found, using first: {matches[0]}")
    
    df = pd.read_csv(matches[0])
    print(f"Loaded: {os.path.basename(matches[0])}")
    return df

# Select datasets to compare
sample_id = '7-19_CitrateBare_hspy'
dataset_1 = {'current': 74, 'stack': 'InSitu_6'}
dataset_2 = {'current': 18, 'stack': 'InSitu_1'}

# Load data
df1 = load_fluence_data(sample_id, dataset_1['current'], dataset_1['stack'])
df2 = load_fluence_data(sample_id, dataset_2['current'], dataset_2['stack'])

if df1 is None or df2 is None:
    raise FileNotFoundError("Could not load one or both datasets")

# Calculate electron dose
df1['electron_dose'] = df1['frame'].apply(
    lambda f: calculate_dose(dataset_1['current'], f, time_per_frame[dataset_1['current']], scan_pixels, pixel_area_nm2)
)
df2['electron_dose'] = df2['frame'].apply(
    lambda f: calculate_dose(dataset_2['current'], f, time_per_frame[dataset_2['current']], scan_pixels, pixel_area_nm2)
)

print(f"\n74 pA dose range: {df1['electron_dose'].min():.2e} to {df1['electron_dose'].max():.2e} e⁻/nm²")
print(f"18 pA dose range: {df2['electron_dose'].min():.2e} to {df2['electron_dose'].max():.2e} e⁻/nm²")

# Get colors and styles from dictionaries
color_74 = sample_colours[sample_id]['74']
color_18 = sample_colours[sample_id]['18']
marker_74 = current_markers[74]
marker_18 = current_markers[18]
linestyle_74 = current_linestyles[74]
linestyle_18 = current_linestyles[18]
label_74 = current_labels[74]
label_18 = current_labels[18]
sample_name = sample_names[sample_id]

stack1_num = dataset_1['stack'].split('_')[-1]
stack2_num = dataset_2['stack'].split('_')[-1]

# Plot: overlay on electron dose
fig, ax = plt.subplots(figsize=(8, 5))

ax.plot(df1['electron_dose'], df1['ce3_fraction'], 
        marker=marker_74, linestyle=linestyle_74, linewidth=2, markersize=6,
        color=color_74, alpha=0.85, label=label_74)

ax.plot(df2['electron_dose'], df2['ce3_fraction'], 
        marker=marker_18, linestyle=linestyle_18, linewidth=2, markersize=6,
        color=color_18, alpha=0.85, label=label_18)

ax.set_xlabel('Electron Fluence (e$^-$/nm$^2$)', fontsize=14, fontweight='bold')
ax.set_ylabel('Ce$^{3+}$ Fraction (edge)', fontsize=14, fontweight='bold')
ax.set_ylim(0.2, 0.4)
ax.set_title(f'{sample_name}', fontsize=14, fontweight='bold',  pad=20)
ax.grid(alpha=0.3, linestyle=':')
ax.legend(fontsize=10, loc='best')
ax.ticklabel_format(style='scientific', axis='x', scilimits=(0,0))
plt.tight_layout()

out_dir = os.path.join(root_dir, sample_id, 'manual')
os.makedirs(out_dir, exist_ok=True)
out_fig = os.path.join(out_dir, f"comparison_{dataset_1['current']}pA_InSitu{stack1_num}_vs_{dataset_2['current']}pA_InSitu{stack2_num}_electron_dose.png")
if SAVE_FIGURES:
    plt.savefig(out_fig, dpi=300, bbox_inches='tight')
plt.show()

# Summary statistics
print("COMPARISON SUMMARY")
print(f"Sample: {sample_name}")
print(f"\n{label_74}:")
print(f"  Frames: {df1['frame'].min()} to {df1['frame'].max()}")
print(f"  Ce3+ range: {df1['ce3_fraction'].min():.3f} to {df1['ce3_fraction'].max():.3f}")
print(f"  Max electron dose: {df1['electron_dose'].max():.2e} e⁻/nm²")

print(f"\n{label_18}:")
print(f"  Frames: {df2['frame'].min()} to {df2['frame'].max()}")
print(f"  Ce3+ range: {df2['ce3_fraction'].min():.3f} to {df2['ce3_fraction'].max():.3f}")
print(f"  Max electron dose: {df2['electron_dose'].max():.2e} e⁻/nm²")

In [ ]:
fntsize = 14
lblsize = 12
sample_id = 'Pd_DecompCeria' # or other sample id
path = r'\ProcessingParams\Pd_DecompCeria'
def load_all_stacks(sample_folder, current_pA, path):
    """Load all fluence CSVs for a given sample and current"""
    pattern = os.path.join(path, f"*{current_pA}pA*fluence_plot_data_rms.csv")
    matches = glob.glob(pattern)
    
    if not matches:
        print(f"No CSVs found for {sample_folder} | {current_pA}pA")
        return []
    
    dfs = []
    for file in matches:
        df = pd.read_csv(file)
        df['stack'] = os.path.basename(file).split('_')[1]  # Extract stack info if needed
        dfs.append(df)
        print(f"Loaded: {os.path.basename(file)}")
    return dfs

def add_electron_dose(df_list, current):
    """Add electron dose column to all dataframes in a list"""
    for df in df_list:
        df['electron_dose'] = df['frame'].apply(
            lambda f: calculate_dose(current, f, time_per_frame[current], scan_pixels, pixel_area_nm2)
        )
    return df_list

# Load all datasets for each current
current_74_dfs = load_all_stacks(sample_id, 74, path)
current_18_dfs = load_all_stacks(sample_id, 18, path)

if not current_74_dfs or not current_18_dfs:
    raise FileNotFoundError("Could not load datasets for one or both currents")

# Add electron dose
current_74_dfs = add_electron_dose(current_74_dfs, 74)
current_18_dfs = add_electron_dose(current_18_dfs, 18)

# Get colors and styles from dictionaries
color_74 = sample_colours[sample_id]['74']
color_18 = sample_colours[sample_id]['18']
marker_74 = current_markers[74]
marker_18 = current_markers[18]
linestyle_74 = current_linestyles[74]
linestyle_18 = current_linestyles[18]
label_74 = current_labels[74]
label_18 = current_labels[18]
sample_name = sample_names[sample_id]

# Plot all stacks per current
fig, ax = plt.subplots(figsize=(8,5))

for df in current_74_dfs:
    stack_label = df['stack'].iloc[0]
    ax.errorbar(df['electron_dose'], df['ce3_fraction'], yerr=df['ce3_fraction_err'],
            marker=marker_74, linestyle=linestyle_74, linewidth=1.5, markersize=5,
            color=color_74, alpha=0.7, label=f"{label_74} {stack_label}", capsize=3)

for df in current_18_dfs:
    stack_label = df['stack'].iloc[0]
    ax.errorbar(df['electron_dose'], df['ce3_fraction'], yerr=df['ce3_fraction_err'],
            marker=marker_18, linestyle=linestyle_18, linewidth=1.5, markersize=5,
            color=color_18, alpha=0.5, label=f"{label_18} {stack_label}", capsize=3)

ax.set_xlabel('Electron Fluence (e$^-$/nm$^2$)', fontsize=fntsize, fontweight='bold')
ax.set_ylabel('Ce$^{3+}$ Fraction (edge)', fontsize=fntsize, fontweight='bold')
ax.tick_params(axis='both', which='major', labelsize=fntsize)
ax.grid(alpha=0.3, linestyle=':')
ax.set_ylim(0.1, 0.4)
ax.legend(fontsize=lblsize, loc='best')
ax.ticklabel_format(style='scientific', axis='x', scilimits=(0,0))
plt.tight_layout()
plt.show()
